In [1]:
import tensorflow as tf                     # type: ignore
from tensorflow.keras import layers, models # type: ignore
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive              # type: ignore
import tensorflowjs as tfjs                 # type: ignore
import os

In [2]:
# Montar y configurar google drive
drive.mount('/content/drive')
ruta = "/content/drive/MyDrive/DigitsRecognition"

Mounted at /content/drive


In [3]:
# Cargar y preprocesar los datos (MINST)
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalizar y redimensionar para la CNN
x_train = x_train.reshape((x_train.shape[0], 28, 28, 1)).astype('float32') / 255
x_test = x_test.reshape((x_test.shape[0], 28, 28, 1)).astype('float32') / 255

# One-hot encoding de las etiquetas
y_train_cat = tf.keras.utils.to_categorical(y_train, 10)
y_test_cat = tf.keras.utils.to_categorical(y_test, 10)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
# Aumento de datos para mejorar generalizacion
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1
)
datagen.fit(x_train)

In [6]:
# Arquitectura del modelo (CNN)
model = models.Sequential([
    # Primera capa convolucional
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1), padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.2), # Regularización para evitar overfitting

    # Segunda capa convolucional
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.3),

    # Capas de aplanado y densas
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    # Capa de salida: 10 unidades (una por dígito), activación Softmax
    layers.Dense(10, activation='softmax')
])

# Mostrar resumen del modelo
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       803,072 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 872,426 (3.33 MB)

 Trainable params: 871,530 (3.32 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
# Guardar el modelo
tfjs.converters.save_keras_model(model, ruta)

In [14]:
# Compilar y entrenar el modelo
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Parámetros de entrenamiento
epochs = 5
batch_size = 128

# Entrenar usando el generador de Data Augmentation
history = model.fit(
    datagen.flow(x_train, y_train_cat, batch_size=batch_size),                   
    epochs=epochs,
    validation_data=(x_test, y_test_cat),
    verbose=1
)

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 319s 672ms/step - accuracy: 0.9844 - loss: 0.0503 - val_accuracy: 0.9855 - val_loss: 0.0440
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 315s 670ms/step - accuracy: 0.9855 - loss: 0.0476 - val_accuracy: 0.9929 - val_loss: 0.0205
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 307s 654ms/step - accuracy: 0.9874 - loss: 0.0412 - val_accuracy: 0.9921 - val_loss: 0.0241
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 313s 668ms/step - accuracy: 0.9886 - loss: 0.0365 - val_accuracy: 0.9953 - val_loss: 0.0153
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 315s 672ms/step - accuracy: 0.9899 - loss: 0.0333 - val_accuracy: 0.9944 - val_loss: 0.0186


In [15]:
# Evaluar el modelo
test_loss, test_acc = model.evaluate(x_test, y_test_cat, verbose=0)
print(f'Precisión final en el conjunto de test (sin augmentation): {test_acc*100:.2f}%')

Precisión final en el conjunto de test (sin augmentation): 99.44%
